In [148]:
# Config: weekly rebalance, 5td forward returns
import sys
sys.path.append("../")
import config
import duckdb
import numpy as np
import pandas as pd
import xgboost as xgb

PARQUET_PATH = str(config.MASTER_FEATURES_PATH)
DATA_START = "2000-01-01"
IS_WEEKS = 364     # ~7 years
OOS_WEEKS = 208    # ~4 years
EMBARGO_WEEKS = 8  # ~2 months
MIN_OOS_WEEKS = 52 # skip fold if OOS would be shorter
LABEL_THRESHOLD = 0.02  # label=1 when fwd_ret_5td > threshold (threshold labels)
TOP_N = 200  # select top N by prediction for portfolio (fixed count)
EXCLUDE = ["ticker", "date", "sector", "famaindustry", "fwd_ret_5td", "label", "fold", "spy_regime_score"]
# Model: "xgboost" | "logistic" | "ridge" | "random_forest" (simpler models often generalize better)
MODEL_TYPE = "xgboost"
FINAL_TEST_WEEKS = 260  # 5-year final test: train on all data before embargo, test on last 5 years (separate from folds)
# Top 25 features by importance (from full-model run); model uses only these
TOP_25_FEATURES = [
    "vix", "nfci", "yield_curve", "hy_spread", "vix_change_20d", "real_rate", "payout_ratio",
"inst_shrholders", "inst_shrvalue", "vol_vs_sector", "ret_12m", "current_ratio", "ret_3m",
    "ret_1m", "inst_shrunits", "ret_6m", "dividend_yield", "capex_intensity", "accrual_ratio",
    "pe_vs_sector", "pb_pit", "pretax_margin", "pcf_pit", "ps_pit", "ps_vs_sector",
]

In [149]:
# DuckDB connection and week-ends from actual trading dates (last trading day per week)
conn = duckdb.connect()
week_ends_df = conn.execute("""
    SELECT MAX(date) AS rebal_date
    FROM read_parquet(?)
    WHERE date >= ?
    GROUP BY year(date), date_part('week', date)
    ORDER BY rebal_date
""", [PARQUET_PATH, DATA_START]).df()
week_ends = week_ends_df["rebal_date"].tolist()
conn.register("week_ends", pd.DataFrame({"rebal_date": week_ends}))
print(f"Week-ends: {len(week_ends)} from {week_ends[0]} to {week_ends[-1]}")

Week-ends: 1304 from 2000-01-07 00:00:00 to 2024-12-31 00:00:00


In [150]:
def generate_folds(week_ends, is_weeks, oos_weeks, embargo_weeks, min_oos_weeks=52):
    """Rolling window folds. Embargo weeks are skipped between IS end and OOS start.
    Skip a fold if its OOS would have fewer than min_oos_weeks."""
    folds = []
    i = 0
    while True:
        oos_start_idx = i + is_weeks + embargo_weeks
        if oos_start_idx >= len(week_ends):
            break
        is_start = week_ends[i]
        is_end = week_ends[i + is_weeks - 1]
        oos_start = week_ends[oos_start_idx]
        oos_end_idx = min(
            oos_start_idx + oos_weeks - 1,
            len(week_ends) - 1,
        )
        n_oos = oos_end_idx - oos_start_idx + 1
        if n_oos >= min_oos_weeks:
            oos_end = week_ends[oos_end_idx]
            folds.append((is_start, is_end, oos_start, oos_end))
        if oos_end_idx >= len(week_ends) - 1:
            break
        i += oos_weeks
    return folds

folds = generate_folds(week_ends, IS_WEEKS, OOS_WEEKS, EMBARGO_WEEKS, MIN_OOS_WEEKS)
for idx, (is_start, is_end, oos_start, oos_end) in enumerate(folds):
    print(f"Fold {idx+1}: IS {is_start.date()} → {is_end.date()} | OOS {oos_start.date()} → {oos_end.date()}")

Fold 1: IS 2000-01-07 → 2006-12-22 | OOS 2007-03-02 → 2011-02-11
Fold 2: IS 2004-01-02 → 2010-12-17 | OOS 2011-02-18 → 2015-02-06
Fold 3: IS 2007-12-31 → 2014-12-19 | OOS 2015-02-13 → 2019-02-08
Fold 4: IS 2011-12-23 → 2018-12-14 | OOS 2019-02-15 → 2023-01-27
Fold 5: IS 2015-12-18 → 2022-12-02 | OOS 2023-02-03 → 2024-12-31


In [151]:
def load_fold(conn, is_start, is_end, oos_start, oos_end):
    """Load one fold: only IS and OOS dates (embargo excluded). Returns DataFrame with fold column."""
    return conn.execute("""
        WITH rebal_dates AS (
            SELECT rebal_date FROM week_ends
            WHERE (rebal_date >= ? AND rebal_date <= ?)
               OR (rebal_date >= ? AND rebal_date <= ?)
        ),
        universe AS (
            SELECT f.ticker, f.date, f.sector, f.famaindustry,
                   f.fwd_ret_5td, f.ret_1m, f.ret_3m, f.ret_6m, f.ret_12m,
                   f.vol_20d, f.vol_60d, f.vol_ratio, f.volume_ratio_1m, f.pct_52w_range,
                   f.ma50_cross, f.ma200_cross, f.atr_14d_normalized, f.momentum_skew_60d,
                   f.ncfo_r2_5y, f.ncfo_cagr_5y, f.ncfo_r2_10y, f.ncfo_cagr_10y,
                   f.ncfo_pct_positive, f.grossmargin_slope, f.capex_intensity,
                   f.accrual_ratio, f.pcf_pit, f.roe, f.current_ratio, f.vix,
                   f.pe_pit, f.pb_pit, f.ps_pit, f.evebitda_pit,
                   f.dividend_yield, f.pretax_margin, f.debt_to_equity, f.liabilities_to_assets,
                   f.payout_ratio, f.earnings_growth_yoy,
                   f.pe_vs_sector, f.pb_vs_sector, f.ps_vs_sector, f.pcf_vs_sector,
                   f.evebitda_vs_sector, f.roic_vs_sector,
                   f.ret_3m_vs_sector, f.vol_vs_sector, f.ret_3m_rank_sector,
                   f.yield_curve, f.hy_spread, f.vix_change_20d, f.nfci, f.real_rate,
                   f.spy_regime_ma, f.spy_ret_1m, f.spy_ret_3m, f.spy_ret_6m, f.spy_ret_12m,
                   f.insider_buy_count_90d, f.insider_sell_count_90d, f.insider_net_shares_90d, f.insider_net_ratio_90d,
                   f.insider_officer_buy_90d,
                   f.inst_shrholders, f.inst_shrunits, f.inst_shrvalue, f.inst_put_call_ratio,
                   f.inst_shrholders_chg_qoq, f.inst_shrunits_chg_qoq,
            FROM read_parquet(?) f
            INNER JOIN rebal_dates r ON f.date = r.rebal_date
            WHERE f.ncfo_r2_5y > 0.5 AND f.fcf_cagr_5y > 0 and f.roe > 0.12 and f.debt_to_equity < 1.5 AND f.grossmargin_slope > 0
            AND f.ret_12m > 0
            AND f.ret_1m > -0.15
            AND f.ret_6m > f.ret_3m
            and (f.vix > 13 and f.vix < 35)
            AND (f.marketcap_daily IS NOT NULL AND f.marketcap_daily >= 2000)
        )
        SELECT *,
               CASE WHEN date <= ? THEN 'is' ELSE 'oos' END AS fold
        FROM universe
        ORDER BY date, ticker
    """, [is_start, is_end, oos_start, oos_end, PARQUET_PATH, is_end]).df()

In [152]:
PERIODS_PER_YEAR = 52  # weekly rebalance
def evaluate_fold(df, preds, top_n=TOP_N):
    """Rank by pred within each date; select top top_n names; equal-weight weekly return (fwd_ret_5td); annualized Sharpe."""
    d = df.copy()
    d["pred"] = preds
    d["rank"] = d.groupby("date")["pred"].rank(ascending=False)
    d["selected"] = d["rank"] <= top_n
    period_rets = (
        d.loc[d["selected"]]
        .groupby("date")["fwd_ret_5td"]
        .mean()
        .rename("port_ret")
    )
    if period_rets.empty or period_rets.std() == 0:
        sharpe = np.nan
    else:
        sharpe = period_rets.mean() / period_rets.std() * (PERIODS_PER_YEAR ** 0.5)
    return {
        "sharpe": sharpe,
        "hit_rate": (period_rets > 0).mean() if len(period_rets) else np.nan,
        "avg_period_ret": period_rets.mean() if len(period_rets) else np.nan,
        "worst_period": period_rets.min() if len(period_rets) else np.nan,
        "n_periods": len(period_rets),
        "monthly_rets": period_rets,  # kept for downstream (oos_monthly key, stitched OOS)
    }

In [153]:
# Per-fold fit and evaluate (limit to top 25 features)
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import RidgeClassifier
from sklearn.ensemble import RandomForestClassifier

base_feature_cols = [c for c in load_fold(conn, *folds[0]).columns if c not in EXCLUDE]
feature_cols = base_feature_cols + ["sector_enc", "famaindustry_enc", "week_of_year"]
print(f"Using {len(feature_cols)} features (incl. encoded sector, famaindustry, week_of_year)")
fold_results = []

# Model factory: XGBoost (strict) or simpler sklearn models (median impute + classifier)
if MODEL_TYPE == "xgboost":
    # Anti-overfitting: shallow trees, strong L1/L2, heavy subsample, early stop
    model_factory = lambda: xgb.XGBClassifier(
        n_estimators=400,
        max_depth=3,
        learning_rate=0.05,
        reg_alpha=2.0,
        reg_lambda=3.0,
        subsample=0.7,
        colsample_bytree=0.7,
        min_child_weight=25,
        random_state=42,
        eval_metric="logloss",
        n_jobs=1,
    )
    EARLY_STOPPING_ROUNDS = 15
elif MODEL_TYPE == "logistic":
    model_factory = lambda: make_pipeline(
        SimpleImputer(strategy="median"),
        LogisticRegression(C=0.1, max_iter=500, random_state=42),
    )
elif MODEL_TYPE == "ridge":
    model_factory = lambda: make_pipeline(
        SimpleImputer(strategy="median"),
        RidgeClassifier(alpha=1.0, random_state=42),
    )
elif MODEL_TYPE == "random_forest":
    model_factory = lambda: make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_leaf=50, random_state=42),
    )
else:
    raise ValueError(f"Unknown MODEL_TYPE: {MODEL_TYPE}")
print(f"Model: {MODEL_TYPE}")

Using 68 features (incl. encoded sector, famaindustry, week_of_year)
Model: xgboost


In [154]:
for fold_idx, (is_start, is_end, oos_start, oos_end) in enumerate(folds):
    df = load_fold(conn, is_start, is_end, oos_start, oos_end)
    # Threshold labels: label=1 when fwd_ret_5td > LABEL_THRESHOLD (NaN stays NaN)
    df["label"] = np.where(df["fwd_ret_5td"].notna(), (df["fwd_ret_5td"] > LABEL_THRESHOLD).astype(int), np.nan)
    is_df = df[df["fold"] == "is"].copy()
    oos_df = df[df["fold"] == "oos"].copy()

    # Encode sector and famaindustry (fit on IS, transform OOS; unseen -> -1)
    cat_cols = ["sector", "famaindustry"]
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    is_cat = is_df[cat_cols].fillna("MISSING").astype(str)
    oos_cat = oos_df[cat_cols].fillna("MISSING").astype(str)
    enc.fit(is_cat)
    is_enc = enc.transform(is_cat)
    oos_enc = enc.transform(oos_cat)
    is_df["sector_enc"] = is_enc[:, 0]
    is_df["famaindustry_enc"] = is_enc[:, 1]
    oos_df["sector_enc"] = oos_enc[:, 0]
    oos_df["famaindustry_enc"] = oos_enc[:, 1]
    is_df["week_of_year"] = pd.to_datetime(is_df["date"]).dt.isocalendar().week.astype(int)
    oos_df["week_of_year"] = pd.to_datetime(oos_df["date"]).dt.isocalendar().week.astype(int)

    X_is = is_df[feature_cols]
    y_is = is_df["label"]
    X_oos = oos_df[feature_cols]
    y_oos = oos_df["label"]
    # Only drop rows with null label; XGBoost handles nulls in features
    mask_is = y_is.notna()
    mask_oos = y_oos.notna()
    X_is, y_is = X_is[mask_is], y_is[mask_is]
    X_oos, y_oos = X_oos[mask_oos], y_oos[mask_oos]
    is_df = is_df.loc[X_is.index]
    oos_df = oos_df.loc[X_oos.index]

    # XGBoost: use last 20% of IS dates as validation for early stopping only; fit on the rest.
    # Non-XGBoost: no early stopping, so fit on full IS to use all data (no train/val split).
    model = model_factory()
    if MODEL_TYPE == "xgboost":
        is_dates = is_df["date"].unique()
        n_val = max(1, int(0.2 * len(is_dates)))
        val_dates = set(sorted(is_dates)[-n_val:])
        val_mask = is_df["date"].isin(val_dates)
        X_tr, X_val = X_is[~val_mask], X_is[val_mask]
        y_tr, y_val = y_is[~val_mask], y_is[val_mask]
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )
    else:
        model.fit(X_is, y_is)
    last_fold_model = model
    is_preds = model.predict_proba(is_df[feature_cols])[:, 1]
    oos_preds = model.predict_proba(oos_df[feature_cols])[:, 1]

    is_metrics = evaluate_fold(is_df, is_preds)
    oos_metrics = evaluate_fold(oos_df, oos_preds)

    fold_results.append({
        "fold": fold_idx + 1,
        "is_start": is_start,
        "is_end": is_end,
        "oos_start": oos_start,
        "oos_end": oos_end,
        "is_sharpe": is_metrics["sharpe"],
        "oos_sharpe": oos_metrics["sharpe"],
        "is_hit_rate": is_metrics["hit_rate"],
        "oos_hit_rate": oos_metrics["hit_rate"],
        "degradation": is_metrics["sharpe"] - oos_metrics["sharpe"],
        "oos_monthly": oos_metrics["monthly_rets"],
    })
    oos_s = oos_metrics["sharpe"]
    n_oos = oos_metrics.get("n_periods", 0)
    oos_str = f"{oos_s:.3f}" if pd.notna(oos_s) else f"n/a ({n_oos} OOS weeks)"
    print(f"Fold {fold_idx+1}: IS Sharpe {is_metrics['sharpe']:.3f}  OOS Sharpe {oos_str}")

Fold 1: IS Sharpe 0.310  OOS Sharpe 0.528
Fold 2: IS Sharpe 0.636  OOS Sharpe 1.022
Fold 3: IS Sharpe 0.790  OOS Sharpe 0.934
Fold 4: IS Sharpe 1.213  OOS Sharpe 0.544
Fold 5: IS Sharpe 0.659  OOS Sharpe 1.183


In [155]:
# Final test (not a fold): train on all data before embargo, test on last FINAL_TEST_WEEKS (5 years)
n = len(week_ends)
if n >= FINAL_TEST_WEEKS + EMBARGO_WEEKS + 1:
    is_end_idx = n - FINAL_TEST_WEEKS - EMBARGO_WEEKS - 1
    is_start = week_ends[0]
    is_end = week_ends[is_end_idx]
    oos_start = week_ends[n - FINAL_TEST_WEEKS]
    oos_end = week_ends[-1]
    print(f"Final test: IS {is_start.date()} → {is_end.date()} | OOS {oos_start.date()} → {oos_end.date()} ({FINAL_TEST_WEEKS} weeks)")

    df = load_fold(conn, is_start, is_end, oos_start, oos_end)
    df["label"] = np.where(df["fwd_ret_5td"].notna(), (df["fwd_ret_5td"] > LABEL_THRESHOLD).astype(int), np.nan)
    is_df = df[df["fold"] == "is"].copy()
    oos_df = df[df["fold"] == "oos"].copy()

    if len(is_df) == 0 or len(oos_df) == 0:
        print(f"Final test skipped: IS rows={len(is_df)}, OOS rows={len(oos_df)} (regime filter may have removed all IS weeks)")
        final_test_result = None
    else:
        cat_cols = ["sector", "famaindustry"]
        enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        is_cat = is_df[cat_cols].fillna("MISSING").astype(str)
        oos_cat = oos_df[cat_cols].fillna("MISSING").astype(str)
        enc.fit(is_cat)
        is_df["sector_enc"] = enc.transform(is_cat)[:, 0]
        is_df["famaindustry_enc"] = enc.transform(is_cat)[:, 1]
        oos_df["sector_enc"] = enc.transform(oos_cat)[:, 0]
        oos_df["famaindustry_enc"] = enc.transform(oos_cat)[:, 1]
        is_df["week_of_year"] = pd.to_datetime(is_df["date"]).dt.isocalendar().week.astype(int)
        oos_df["week_of_year"] = pd.to_datetime(oos_df["date"]).dt.isocalendar().week.astype(int)

        X_is = is_df[feature_cols]
        y_is = is_df["label"]
        X_oos = oos_df[feature_cols]
        y_oos = oos_df["label"]
        mask_is = y_is.notna()
        mask_oos = y_oos.notna()
        X_is, y_is = X_is[mask_is], y_is[mask_is]
        X_oos, y_oos = X_oos[mask_oos], y_oos[mask_oos]
        is_df = is_df.loc[X_is.index]
        oos_df = oos_df.loc[X_oos.index]

        model = model_factory()
        if MODEL_TYPE == "xgboost":
            is_dates = is_df["date"].unique()
            n_val = max(1, int(0.2 * len(is_dates)))
            val_dates = set(sorted(is_dates)[-n_val:])
            val_mask = is_df["date"].isin(val_dates)
            X_tr, X_val = X_is[~val_mask], X_is[val_mask]
            y_tr, y_val = y_is[~val_mask], y_is[val_mask]
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        else:
            model.fit(X_is, y_is)

        is_preds = model.predict_proba(is_df[feature_cols])[:, 1]
        oos_preds = model.predict_proba(oos_df[feature_cols])[:, 1]
        is_metrics = evaluate_fold(is_df, is_preds)
        oos_metrics = evaluate_fold(oos_df, oos_preds)

        final_test_result = {
            "is_start": is_start, "is_end": is_end, "oos_start": oos_start, "oos_end": oos_end,
            "is_sharpe": is_metrics["sharpe"], "oos_sharpe": oos_metrics["sharpe"],
            "is_hit_rate": is_metrics["hit_rate"], "oos_hit_rate": oos_metrics["hit_rate"],
        }
        print(f"Final test: IS Sharpe {is_metrics['sharpe']:.3f}  OOS Sharpe {oos_metrics['sharpe']:.3f}  OOS hit rate {oos_metrics['hit_rate']:.3f}")
        final_model = model
        final_encoder = enc
else:
    print(f"Not enough week-ends for final test (need {FINAL_TEST_WEEKS + EMBARGO_WEEKS + 1}, have {n})")
    final_test_result = None

Final test: IS 2000-01-07 → 2019-11-15 | OOS 2020-01-10 → 2024-12-31 (260 weeks)
Final test: IS Sharpe 0.624  OOS Sharpe 0.674  OOS hit rate 0.557


In [156]:
# Summary DataFrame
summary = pd.DataFrame([
    {
        "fold": f["fold"],
        "is_start": f["is_start"],
        "is_end": f["is_end"],
        "oos_start": f["oos_start"],
        "oos_end": f["oos_end"],
        "is_sharpe": f["is_sharpe"],
        "oos_sharpe": f["oos_sharpe"],
        "is_hit_rate": f["is_hit_rate"],
        "oos_hit_rate": f["oos_hit_rate"],
        "degradation": f["degradation"],
        "oos_positive": f["oos_sharpe"] > 0,
    }
    for f in fold_results
])
summary

,fold,is_start,is_end,oos_start,oos_end,is_sharpe,oos_sharpe,is_hit_rate,oos_hit_rate,degradation,oos_positive
0,1,2000-01-07,2006-12-22,2007-03-02,2011-02-11,0.310176,0.527676,0.544061,0.608434,-0.217500,True
1,2,2004-01-02,2010-12-17,2011-02-18,2015-02-06,0.636314,1.021569,0.601660,0.580247,-0.385255,True
2,3,2007-12-31,2014-12-19,2015-02-13,2019-02-08,0.789930,0.933554,0.595070,0.577236,-0.143624,True
3,4,2011-12-23,2018-12-14,2019-02-15,2023-01-27,1.213329,0.543578,0.591667,0.559322,0.669751,True
4,5,2015-12-18,2022-12-02,2023-02-03,2024-12-31,0.659404,1.183334,0.556420,0.578313,-0.523930,True


In [157]:
# Persist final model for later inference (run after Final test cell)
if final_test_result is not None and "final_model" in dir():
    import joblib
    out_path = config.OUTPUTS_DIR / "walk_forward_final_model.joblib"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(
        {"model": final_model, "encoder": final_encoder, "feature_cols": feature_cols},
        out_path,
    )
    print(f"Saved model, encoder, and feature_cols to {out_path}")
else:
    print("Run the Final test cell first to persist the model.")

Saved model, encoder, and feature_cols to /Users/zacharyfrederick/regime/outputs/walk_forward_final_model.joblib


In [158]:
# Score current inference with final model (requires: run Final test cell first; build current_inference.parquet via scripts/build_current_inference.py)
if final_test_result is None or 'final_model' not in dir():
    print("Run the Final test cell first.")
else:
    from pathlib import Path
    current_path = config.OUTPUTS_DIR / "current_inference.parquet"
    if not current_path.exists():
        print(f"Current inference file not found: {current_path}. Run: python scripts/build_current_inference.py")
    else:
        current = pd.read_parquet(current_path)
        if current.empty:
            print("Current inference parquet is empty.")
        else:
            cat_cols = ["sector", "famaindustry"]
            current_cat = current[cat_cols].fillna("MISSING").astype(str)
            current["sector_enc"] = final_encoder.transform(current_cat)[:, 0]
            current["famaindustry_enc"] = final_encoder.transform(current_cat)[:, 1]
            current["week_of_year"] = pd.to_datetime(current["date"]).dt.isocalendar().week.fillna(1).astype(int)
            for col in feature_cols:
                if col not in current.columns:
                    current[col] = np.nan
            X_current = current[feature_cols].copy()
            for c in X_current.select_dtypes(include=["object"]).columns:
                X_current[c] = pd.to_numeric(X_current[c], errors="coerce")
            current["score"] = final_model.predict_proba(X_current)[:, 1]
            n_top = min(TOP_N, len(current))
            print(f"Current inference: {len(current)} tickers, top {n_top} names")
            display(current.sort_values("score", ascending=False)[["ticker", "score", "sector", "famaindustry"]].head(50))

Current inference: 1971 tickers, top 200 names


,ticker,score,sector,famaindustry
708,FLNC,0.356659,Utilities,Electrical Equipment
1653,STUB,0.353694,Communication Services,Entertainment
940,IONS,0.341725,Healthcare,Pharmaceutical Products
1475,RGC,0.341305,Healthcare,Pharmaceutical Products
776,GH,0.339597,Healthcare,Healthcare
1303,OKLO,0.339068,Utilities,Utilities
154,ASTS,0.331715,Technology,Communication
815,GSAT,0.331536,Communication Services,Communication
1536,SATS,0.331307,Technology,Electronic Equipment
1774,TVTX,0.330828,Healthcare,Pharmaceutical Products


In [159]:
# Stitched OOS returns and full-period Sharpe
# Reindex to full OOS calendar so weeks with no positions = 0% return
first_oos = fold_results[0]["oos_start"]
last_oos = fold_results[-1]["oos_end"]
full_oos_calendar = pd.DatetimeIndex([d for d in week_ends if first_oos <= d <= last_oos])
pieces = []
for f in fold_results:
    fold_oos_dates = [d for d in week_ends if f["oos_start"] <= d <= f["oos_end"]]
    s = f["oos_monthly"].reindex(fold_oos_dates, fill_value=0.0)
    pieces.append(s)
all_oos_rets = pd.concat(pieces).sort_index()
all_oos_rets = all_oos_rets.reindex(full_oos_calendar, fill_value=0.0)
assert all_oos_rets.index.is_unique, "Overlapping OOS periods detected"
cumulative = (1 + all_oos_rets).cumprod()
full_oos_sharpe = all_oos_rets.mean() / all_oos_rets.std() * (PERIODS_PER_YEAR ** 0.5) if all_oos_rets.std() > 0 else np.nan
n_traded = (all_oos_rets != 0).sum()
print(f"Mean IS Sharpe:    {summary['is_sharpe'].mean():.2f}")
print(f"Mean OOS Sharpe:   {summary['oos_sharpe'].mean():.2f}")
print(f"Mean degradation:  {summary['degradation'].mean():.2f}")
print(f"OOS folds positive: {(summary['oos_sharpe'] > 0).sum()} / {len(summary)}")
print(f"Full OOS Sharpe:   {full_oos_sharpe:.2f}  (all {len(full_oos_calendar)} weeks; {n_traded} traded)")

Mean IS Sharpe:    0.72
Mean OOS Sharpe:   0.84
Mean degradation:  -0.12
OOS folds positive: 5 / 5
Full OOS Sharpe:   0.68  (all 932 weeks; 711 traded)


In [160]:
# Diagnostics: degradation ratio, negative OOS, contiguity
for _, row in summary.iterrows():
    if pd.notna(row["oos_sharpe"]) and row["oos_sharpe"] != 0:
        ratio = row["is_sharpe"] / row["oos_sharpe"]
        if ratio > 3:
            print(f"Warning: Fold {row['fold']} degradation ratio {ratio:.1f}x > 3")
n_neg = (summary["oos_sharpe"] <= 0).sum()
if n_neg:
    print(f"Flag: {n_neg} fold(s) with non-positive OOS Sharpe")
assert all_oos_rets.index.is_unique
# Contiguity: gap between fold N oos_end and fold N+1 oos_start at most one week
for i in range(len(fold_results) - 1):
    end_prev = fold_results[i]["oos_end"]
    start_next = fold_results[i + 1]["oos_start"]
    weeks_diff = (start_next - end_prev).days / 7.0
    assert weeks_diff <= 1.5, f"Gap between fold {i+1} and {i+2} OOS: {weeks_diff:.1f} weeks"
print("Diagnostics OK: no duplicate OOS dates, OOS periods contiguous.")

Diagnostics OK: no duplicate OOS dates, OOS periods contiguous.


In [161]:
# Feature importances (from last fold's model)
if MODEL_TYPE == "xgboost" and "last_fold_model" in dir():
    imp = last_fold_model.feature_importances_
    importance_df = pd.DataFrame({"feature": feature_cols, "importance": imp}).sort_values("importance", ascending=False)
    print("Feature importances (last fold):")
    print(importance_df.to_string(index=False))
elif MODEL_TYPE == "random_forest" and "last_fold_model" in dir():
    imp = last_fold_model.steps[-1][1].feature_importances_
    importance_df = pd.DataFrame({"feature": feature_cols, "importance": imp}).sort_values("importance", ascending=False)
    print("Feature importances (last fold):")
    print(importance_df.to_string(index=False))
elif MODEL_TYPE in ("logistic", "ridge") and "last_fold_model" in dir():
    clf = last_fold_model.steps[-1][1]
    coef = np.abs(clf.coef_).ravel()
    importance_df = pd.DataFrame({"feature": feature_cols, "importance": coef}).sort_values("importance", ascending=False)
    print("Feature importances (|coef|, last fold):")
    print(importance_df.to_string(index=False))
else:
    print("Run the fold loop first; feature importances available for xgboost, random_forest, logistic, ridge.")

Feature importances (last fold):
                feature  importance
                    vix    0.049924
             spy_ret_1m    0.033410
                vol_60d    0.032567
            yield_curve    0.028717
           week_of_year    0.028592
         vix_change_20d    0.028323
             spy_ret_3m    0.025382
             spy_ret_6m    0.025354
                vol_20d    0.025040
            spy_ret_12m    0.023422
                   nfci    0.022875
              real_rate    0.021491
              hy_spread    0.021359
           ncfo_cagr_5y    0.018126
 insider_sell_count_90d    0.015519
         dividend_yield    0.014895
                 pe_pit    0.014825
           evebitda_pit    0.014441
                ret_12m    0.014410
     atr_14d_normalized    0.014208
             ma50_cross    0.014204
                 pb_pit    0.014064
          vol_vs_sector    0.013786
          inst_shrvalue    0.013658
                pcf_pit    0.013579
                 ps_pit    0.01

In [162]:
current['score'].describe()

count    1971.000000
mean        0.201213
std         0.050325
min         0.061925
25%         0.166361
50%         0.200548
75%         0.237042
max         0.356659
Name: score, dtype: float64

In [163]:
oos_df

,ticker,date,sector,famaindustry,fwd_ret_5td,ret_1m,ret_3m,ret_6m,ret_12m,vol_20d,...,inst_shrunits,inst_shrvalue,inst_put_call_ratio,inst_shrholders_chg_qoq,inst_shrunits_chg_qoq,fold,label,sector_enc,famaindustry_enc,week_of_year
34754,ACGL,2020-01-24,Financial Services,Insurance,-0.018441,0.069520,0.086964,0.183644,0.562144,0.123653,...,3.255578e+08,13660.0,1.218512,21.0,-734019.0,oos,0.0,5.0,23.0,4
34755,AKAM,2020-01-24,Technology,Business Services,-0.022718,0.111861,0.058745,0.155018,0.479325,0.161401,...,1.457949e+08,13315.0,1.061374,24.0,-358462.0,oos,0.0,9.0,7.0,4
34756,AMN,2020-01-24,Healthcare,Business Services,0.006723,0.088470,0.155161,0.203777,0.052358,0.147932,...,4.700481e+07,2705.0,0.727628,-4.0,556239.0,oos,0.0,6.0,7.0,4
34757,ANSS,2020-01-24,Technology,Business Services,-0.010853,0.084503,0.283506,0.300600,0.754206,0.122026,...,8.024823e+07,17757.0,2.335707,8.0,332739.0,oos,0.0,9.0,7.0,4
34758,CASY,2020-01-24,Consumer Cyclical,Retail,-0.026691,0.050433,0.015987,0.031444,0.246572,0.155659,...,3.219204e+07,5187.0,0.527613,1.0,-338937.0,oos,0.0,2.0,37.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48784,TXRH,2024-12-31,Consumer Cyclical,Restaraunts Hotels Motels,0.008978,-0.118235,0.021328,0.083809,0.497518,0.230827,...,6.246230e+07,11028.0,0.709943,43.0,855857.0,oos,0.0,2.0,36.0,1
48785,V,2024-12-31,Financial Services,Business Services,-0.010885,0.003049,0.140642,0.182849,0.223219,0.161962,...,1.524493e+09,416309.0,0.942430,38.0,4426478.0,oos,0.0,5.0,7.0,1
48786,VCTR,2024-12-31,Financial Services,Trading,-0.032538,-0.051487,0.203480,0.383629,0.959882,0.310278,...,5.018932e+07,2787.0,0.324841,7.0,1226343.0,oos,0.0,5.0,44.0,1
48787,VEEV,2024-12-31,Healthcare,Business Services,0.033532,-0.077244,0.012667,0.149850,0.092094,0.451471,...,1.279856e+08,26862.0,1.097311,38.0,238872.0,oos,1.0,6.0,7.0,1


In [164]:
for ticker in current.sort_values(by='score', ascending=False)[:200]['ticker']:
    print(ticker)

FLNC
STUB
IONS
RGC
GH
OKLO
ASTS
GSAT
SATS
TVTX
AAL
CPRI
RYTM
TMC
ALK
RAL
NIQ
INSM
SNAP
ARWR
HRI
BBAI
AXSM
IRTC
MDGL
HUT
ABG
KMX
CVNA
IMNM
ARQT
TEAM
FOUR
AERO
QBTS
GPI
WK
COTY
GLXY
COMP
NTSK
VRDN
EXK
ADPT
GLOB
CRWV
IE
GMAB
BB
GPCR
TARS
CAR
CNTA
BBIO
JOBY
U
GTLB
BL
AVTR
WDAY
W
GT
ALNY
LPLA
RARE
SANM
RCUS
CIFR
AMBA
PII
SWKS
SYM
QS
OPEN
RITM
DKNG
MBLY
F
UUUU
INTC
HUM
UWMC
LYFT
LQDA
BMNR
WULF
CHYM
APLD
OSCR
AN
PVH
RBRK
SCZM
TPG
NAVN
CNXC
NTNX
QTWO
S
STNE
LMND
ONC
TGB
MICC
FMCC
FNMA
LAD
BBUC
MLYS
PJT
PTCT
WSC
CAI
ATKR
FSLR
RUM
ELV
PI
MIR
CVI
HP
CTSH
BRBR
RH
SNOW
DJT
TLN
CLS
AMRX
CLF
ALB
FG
SHC
CHA
CYTK
RUN
SAIL
WHR
DNOW
SMMT
BRKR
VRNS
ACMR
APLS
M
GRFS
FLUT
DXC
COGT
SOFI
CORZ
DB
RGTI
PAY
TTAM
AVPT
CORT
KLAR
BTU
MAZE
HGV
VERX
MDLN
ONDS
NKE
AAP
ARES
DOO
TEVA
PSTG
ARIS
LAZ
NRG
GTM
APTV
MMYT
EQPT
VRRM
SEB
PRM
CRNX
AGYS
YMM
FRSH
KVYO
CWK
BBU
MIAX
FIGR
CPNG
RBLX
ZS
APO
VISN
EE
SNDK
CHWY
MLCO
CSGP
BRZE


In [165]:
current[current['ticker'] == 'NVDA']

,ticker,date,marketcap_daily,sector,famaindustry,fwd_ret_5td,ret_1m,ret_3m,ret_6m,ret_12m,...,insider_net_shares_90d,insider_net_ratio_90d,insider_officer_buy_90d,inst_shrholders,inst_shrunits,inst_shrvalue,inst_put_call_ratio,inst_shrholders_chg_qoq,inst_shrunits_chg_qoq,score
1276,NVDA,None,4321026,Technology,Electronic Equipment,NaN,0.020839,-0.009801,0.036,0.516343,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.205405
